# Live monitoring (separate Workspace tab)

Run while `workspace_parallel_spcs_demo.ipynb` is busy. **FQNs** come from the same YAML as the other notebooks.


## 1. Session + load `parallel_lab`


In [ ]:
import parallel_lab_config as plc
plc.ensure_notebook_path_on_sys_path()
LAB = plc.load_parallel_lab_dict()
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print("LAB database:", LAB["database"])
print(session.get_current_database(), session.get_current_schema())


## 2. Job manifest (aggregated)


In [ ]:
jm = plc.fq_schema_table(LAB, "config", "JOB_MANIFEST")
df = session.sql(f"""
SELECT JOB_ID, JOB_TYPE, STATUS, COUNT(*) AS N,
       MIN(STARTED_AT) AS FIRST_START, MAX(COMPLETED_AT) AS LAST_COMPLETE
FROM {jm}
GROUP BY 1,2,3 ORDER BY FIRST_START DESC NULLS LAST, JOB_ID, STATUS
""").to_pandas()
df


## 3. Training metrics (latest)


In [ ]:
tr = plc.fq_schema_table(LAB, "models", "TRAINING_RESULTS")
df2 = session.sql(f"""
SELECT JOB_ID, UNIT_ID, WORKER_INDEX, RMSE, MAE, AIC, TRAINING_SECS, COMPLETED_AT
FROM {tr}
ORDER BY COMPLETED_AT DESC LIMIT 200
""").to_pandas()
df2


## 4. doSnowflake queue (optional)


In [ ]:
qq = plc.queue_fqn(LAB)
try:
    _q = session.sql(f"""
      SELECT JOB_ID, STATUS, COUNT(*) AS N
      FROM {qq}
      GROUP BY 1,2 ORDER BY 1,2
    """).to_pandas()
    _q
except Exception as e:
    print("Queue table not present or not accessible:", e)
